# 📊 Poll Results Visualizer
### End-to-End Survey Data Analysis Project

**Author:** Anupam Santra  
**Date:** 2024  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn

---
This notebook walks through every step of analyzing synthetic poll data — from dataset creation through cleaning, EDA, analysis, and visualization.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime, timedelta

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
%matplotlib inline

print('Libraries loaded ✅')
print(f'Pandas  : {pd.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'Seaborn : {sns.__version__}')

## 2. Generate Synthetic Poll Dataset

In [ ]:
np.random.seed(42)
N = 500

REGIONS    = ['North', 'South', 'East', 'West', 'Central']
AGE_GROUPS = ['18-25', '26-35', '36-50', '51+']
GENDERS    = ['Male', 'Female', 'Non-binary']
OPTIONS    = ['Option A', 'Option B', 'Option C', 'Option D']
EDUCATION  = ["High School", "Bachelor's", "Master's", "PhD"]
EMPLOYMENT = ['Employed', 'Student', 'Self-employed', 'Unemployed']

REGION_BIAS = {
    'North':   [0.40, 0.30, 0.20, 0.10],
    'South':   [0.25, 0.40, 0.25, 0.10],
    'East':    [0.30, 0.25, 0.30, 0.15],
    'West':    [0.20, 0.25, 0.35, 0.20],
    'Central': [0.35, 0.30, 0.20, 0.15],
}
AGE_BIAS = {
    '18-25': [0.20, 0.25, 0.40, 0.15],
    '26-35': [0.30, 0.30, 0.25, 0.15],
    '36-50': [0.40, 0.30, 0.20, 0.10],
    '51+':   [0.45, 0.30, 0.15, 0.10],
}

base = datetime(2024, 1, 1)
rows = []
for i in range(1, N + 1):
    region    = np.random.choice(REGIONS)
    age_group = np.random.choice(AGE_GROUPS, p=[0.25, 0.30, 0.27, 0.18])
    gender    = np.random.choice(GENDERS, p=[0.48, 0.48, 0.04])
    education = np.random.choice(EDUCATION, p=[0.25, 0.45, 0.22, 0.08])
    employment= np.random.choice(EMPLOYMENT, p=[0.55, 0.20, 0.15, 0.10])
    rb = np.array(REGION_BIAS[region])
    ab = np.array(AGE_BIAS[age_group])
    bias = (rb * 0.6 + ab * 0.4); bias /= bias.sum()
    option = np.random.choice(OPTIONS, p=bias)
    date   = base + timedelta(days=np.random.randint(0, 90))
    rows.append({'respondent_id': f'R{i:04d}', 'date': date.strftime('%Y-%m-%d'),
                 'question': 'Which option do you prefer for the upcoming election?',
                 'selected_option': option, 'region': region, 'age_group': age_group,
                 'gender': gender, 'education': education, 'employment': employment})

df_raw = pd.DataFrame(rows)
noise_idx = np.random.choice(df_raw.index, size=10, replace=False)
df_raw.loc[noise_idx, 'selected_option'] = np.nan

os.makedirs('data', exist_ok=True)
df_raw.to_csv('data/poll_data.csv', index=False)
print(f'Dataset shape: {df_raw.shape}')
df_raw.head(10)

## 3. Data Cleaning

In [ ]:
df = df_raw.copy()
print('Missing values before cleaning:')
print(df.isnull().sum())

# Impute missing values using region mode
region_modes = (df.dropna(subset=['selected_option'])
                  .groupby('region')['selected_option']
                  .agg(lambda x: x.mode()[0]))
missing_mask = df['selected_option'].isna()
df.loc[missing_mask, 'selected_option'] = df.loc[missing_mask, 'region'].map(region_modes)

# Standardise text
for col in ['selected_option','region','age_group','gender','education','employment']:
    df[col] = df[col].astype(str).str.strip().str.title()

# Parse dates and add time features
df['date']  = pd.to_datetime(df['date'])
df['week']  = df['date'].dt.isocalendar().week.astype(int)
df['month'] = df['date'].dt.month_name()

# Age as ordered categorical
age_order = ['18-25', '26-35', '36-50', '51+']
df['age_group'] = pd.Categorical(df['age_group'], categories=age_order, ordered=True)

df.to_csv('data/poll_data_clean.csv', index=False)
print('\nAfter cleaning — missing values:')
print(df.isnull().sum())
df.head()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
print('Dataset Info')
print('─' * 40)
print(f'Shape         : {df.shape}')
print(f'Date range    : {df.date.min().date()} → {df.date.max().date()}')
print(f'Regions       : {df.region.unique().tolist()}')
print(f'Options       : {df.selected_option.unique().tolist()}')
print()
print('Value Counts per column:')
for col in ['selected_option','region','age_group','gender']:
    print(f'  {col}:')
    print(df[col].value_counts().to_string(), '\n')

## 5. Overall Vote Share Analysis

In [ ]:
PALETTE   = ['#2563EB', '#16A34A', '#DC2626', '#D97706']
OPTIONS   = sorted(df['selected_option'].unique())
OPTION_CLR= {o: PALETTE[i] for i, o in enumerate(OPTIONS)}
TOTAL     = len(df)

overall = df['selected_option'].value_counts().reset_index()
overall.columns = ['option', 'votes']
overall['share_%'] = (overall['votes'] / TOTAL * 100).round(2)
print(overall.to_string(index=False))

## 6. Visualizations

In [ ]:
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#E5E7EB', 'figure.dpi': 100})
os.makedirs('outputs', exist_ok=True)

# --- Chart 1: Horizontal bar ---
data = overall.sort_values('share_%')
fig, ax = plt.subplots(figsize=(9, 4))
colors = [OPTION_CLR[o] for o in data['option']]
bars = ax.barh(data['option'], data['share_%'], color=colors, height=0.5, edgecolor='white')
for bar, val in zip(bars, data['share_%']):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2, f'{val:.1f}%',
            va='center', fontsize=11, fontweight='bold')
ax.set_title('Overall Poll Results — Vote Share', fontsize=14, fontweight='bold')
ax.set_xlabel('Vote Share (%)')
plt.tight_layout()
plt.savefig('outputs/chart1_overall_bar.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Chart 1 saved ✅')

In [ ]:
# --- Chart 2: Donut ---
counts = df['selected_option'].value_counts()
fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(counts.values, labels=counts.index,
    autopct='%1.1f%%', colors=[OPTION_CLR[o] for o in counts.index],
    startangle=140, pctdistance=0.82,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2))
for t in autotexts:
    t.set_fontsize(12); t.set_fontweight('bold'); t.set_color('white')
centre = plt.Circle((0,0), 0.45, color='white')
ax.add_artist(centre)
ax.text(0, 0.06, f'{TOTAL}', ha='center', va='center', fontsize=22, fontweight='bold')
ax.text(0, -0.14, 'Responses', ha='center', fontsize=11, color='gray')
ax.set_title('Vote Share — Donut Chart', fontsize=14, fontweight='bold')
plt.savefig('outputs/chart2_donut.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Chart 2 saved ✅')

In [ ]:
# --- Chart 3: Region-wise grouped bar ---
pivot = df.groupby(['region','selected_option']).size().unstack(fill_value=0)
pct   = pivot.div(pivot.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(11, 5))
pct.plot(kind='bar', ax=ax, color=PALETTE, width=0.75, edgecolor='white')
ax.set_title('Region-wise Poll Results', fontsize=14, fontweight='bold')
ax.set_xlabel('Region'); ax.set_ylabel('Vote Share (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Option', bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout()
plt.savefig('outputs/chart3_region_grouped.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Chart 3 saved ✅')

In [ ]:
# --- Chart 4: Age stacked ---
age_order = ['18-25','26-35','36-50','51+']
pivot = df.groupby(['age_group','selected_option']).size().unstack(fill_value=0).reindex(age_order)
pct   = pivot.div(pivot.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(9, 5))
bottom = np.zeros(len(pct))
for i, col in enumerate(pct.columns):
    bars = ax.bar(pct.index, pct[col], bottom=bottom, color=PALETTE[i], label=col, edgecolor='white')
    for bar, val in zip(bars, pct[col]):
        if val > 5:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_y()+bar.get_height()/2,
                    f'{val:.0f}%', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    bottom += pct[col].values
ax.set_title('Age-Group Breakdown — Stacked Bar', fontsize=14, fontweight='bold')
ax.set_xlabel('Age Group'); ax.set_ylabel('Vote Share (%)')
ax.legend(title='Option', bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout()
plt.savefig('outputs/chart4_age_stacked.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Chart 4 saved ✅')

In [ ]:
# --- Chart 5: Gender clustered bar ---
pivot = df.groupby(['gender','selected_option']).size().unstack(fill_value=0)
pct   = pivot.div(pivot.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(9, 5))
pct.plot(kind='bar', ax=ax, color=PALETTE, width=0.7, edgecolor='white')
ax.set_title('Gender-wise Poll Results', fontsize=14, fontweight='bold')
ax.set_xlabel('Gender'); ax.set_ylabel('Vote Share (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Option', bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout()
plt.savefig('outputs/chart5_gender.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Chart 5 saved ✅')

In [ ]:
# --- Chart 6: Weekly trend ---
trend = df.groupby(['week','selected_option']).size().reset_index(name='votes')
totals= df.groupby('week').size().reset_index(name='total')
trend = trend.merge(totals, on='week')
trend['share'] = trend['votes'] / trend['total'] * 100

fig, ax = plt.subplots(figsize=(11,5))
for opt in sorted(df['selected_option'].unique()):
    sub = trend[trend['selected_option']==opt].sort_values('week')
    ax.plot(sub['week'], sub['share'], marker='o', markersize=5,
            linewidth=2, label=opt, color=OPTION_CLR[opt])
ax.set_title('Weekly Trend — Vote Share Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Week Number (2024)'); ax.set_ylabel('Vote Share (%)')
ax.legend(title='Option')
plt.tight_layout()
plt.savefig('outputs/chart6_weekly_trend.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Chart 6 saved ✅')

In [ ]:
# --- Chart 7: Heatmap ---
pivot = df.groupby(['region','selected_option']).size().unstack(fill_value=0)
pct   = pivot.div(pivot.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(8,5))
sns.heatmap(pct, annot=True, fmt='.1f', cmap='Blues',
            linewidths=0.5, linecolor='#E5E7EB',
            cbar_kws={'label':'Vote Share (%)'}, ax=ax)
ax.set_title('Heatmap: Region × Option — Vote Share (%)', fontsize=14, fontweight='bold')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('outputs/chart7_heatmap.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Chart 7 saved ✅')

## 7. Key Insights Summary

In [ ]:
winner = overall.iloc[0]
runner = overall.iloc[1]
margin = round(winner['share_%'] - runner['share_%'], 2)

leaders = (df.groupby(['region','selected_option']).size()
             .reset_index(name='votes').sort_values('votes', ascending=False)
             .groupby('region').first().reset_index())

print('=' * 55)
print('           KEY INSIGHTS')
print('=' * 55)
print(f'  ✅ Leading option : {winner["option"]} ({winner["share_%"]}% — {winner["votes"]} votes)')
print(f'  🥈 Runner-up      : {runner["option"]} ({runner["share_%"]}%, margin: {margin}pp)')
print(f'  📊 Total responses: {TOTAL}')
print(f'  📅 Survey period  : Jan–Mar 2024')
print()
print('  Regional leaders:')
for _, row in leaders.iterrows():
    print(f'    {row["region"]:10s} → {row["selected_option"]} ({row["votes"]} votes)')
print('=' * 55)

---
## Conclusions

- **Option A** is the overall leader with **32.8%** of total votes.
- **North region** shows the strongest support for Option A (46%).
- **West region** uniquely favours Option C (41.35%), showing regional variation.
- **Older respondents (36–50)** lean more toward Option A, while **younger voters (18–35)** are more evenly distributed.
- The weekly trend shows **consistent leads** for Option A throughout the survey period.
- **Option D** is the least preferred across all demographic segments.

This analysis demonstrates how demographic and regional segmentation can reveal hidden patterns in survey data — a critical skill for Data Analysts and Research Analysts.